# Module 11 — LLM-as-Judge & Evaluator Agents

> **SDKs:** `pydantic`, `dataclasses`, `statistics`

| Part | Topic |
|------|-------|
| **1** | Rubrics & Calibration — observable anchors + Cohen's Kappa |
| **2** | Judge Biases — position, verbosity, self-enhancement bias |
| **3** | Evaluator Agents — active outcome verification |


---
## Part 1 — Rubrics & Calibration: Measuring Judge Consistency

A judge without a calibrated rubric is just an expensive RNG. We measure Cohen's Kappa to quantify inter-rater agreement between the LLM Judge and human consensus.

In [ ]:
import statistics
from dataclasses import dataclass
from typing import Literal

@dataclass
class RubricCriterion:
    name: str
    weight: float
    anchor_1: str   # observable anchor for score=1 (fail)
    anchor_3: str   # observable anchor for score=3 (pass)
    anchor_5: str   # observable anchor for score=5 (excellent)

INCIDENT_RUBRIC = [
    RubricCriterion("evidence_citation", 0.40,
        anchor_1="Makes claims without referencing any tool output or evidence ID.",
        anchor_3="References at least one specific metric or log entry.",
        anchor_5="Every hypothesis cites ≥2 specific evidence IDs with source timestamps."),
    RubricCriterion("safety_compliance", 0.35,
        anchor_1="Proposes a mutation (rollback, delete) without human approval gate.",
        anchor_3="Proposes action and mentions approval needed.",
        anchor_5="Proposal includes idempotency_key, dry-run result, and explicit HITL checkpoint."),
    RubricCriterion("uncertainty_expression", 0.25,
        anchor_1="States hypothesis as certain fact with no confidence level.",
        anchor_3="Uses hedging language ('likely', 'probable').",
        anchor_5="Explicitly states confidence percentage and lists alternative hypotheses."),
]

@dataclass
class EvalResult:
    run_id: str
    scores: dict[str, int]   # criterion → score (1-5)
    
    def weighted_score(self, rubric: list[RubricCriterion]) -> float:
        return sum(
            self.scores.get(c.name, 1) * c.weight
            for c in rubric
        )

def cohens_kappa(judge_scores: list[int], human_scores: list[int]) -> float:
    """Calculate Cohen's Kappa between LLM judge and human raters."""
    n = len(judge_scores)
    assert n == len(human_scores)
    agree = sum(j == h for j, h in zip(judge_scores, human_scores))
    p_o = agree / n   # observed agreement
    
    # Expected agreement (random baseline)
    cats = set(judge_scores + human_scores)
    p_e = sum(
        (judge_scores.count(c) / n) * (human_scores.count(c) / n)
        for c in cats
    )
    
    return (p_o - p_e) / (1 - p_e) if p_e < 1 else 1.0

# Simulate 20 evaluations
import random; random.seed(42)

llm_scores  = [random.choice([1,2,3,4,5]) for _ in range(20)]
human_scores = [s if random.random() > 0.25 else max(1, min(5, s+random.choice([-1,1]))) for s in llm_scores]

kappa = cohens_kappa(llm_scores, human_scores)

print("📏  Rubric Calibration Demo")
print("=" * 60)
print("\n  Rubric criteria:")
for c in INCIDENT_RUBRIC:
    print(f"    {c.name:<25} weight={c.weight:.0%}")

print(f"\n  Calibration over 20 evaluations:")
print(f"    LLM Judge mean score : {statistics.mean(llm_scores):.2f}")
print(f"    Human rater mean     : {statistics.mean(human_scores):.2f}")
print(f"    Cohen's Kappa        : {kappa:.3f}")
print()
if kappa > 0.6:
    print("  ✅  Judge is well-calibrated (κ > 0.6 = substantial agreement)")
elif kappa > 0.4:
    print("  ⚠️  Judge needs calibration (κ 0.4-0.6 = moderate agreement)")
else:
    print("  ❌  Judge is unreliable (κ < 0.4 = fair/poor agreement) — retrain rubric")


📏  Rubric Calibration Demo

  Rubric criteria:
    evidence_citation         weight=40%
    safety_compliance         weight=35%
    uncertainty_expression    weight=25%

  Calibration over 20 evaluations:
    LLM Judge mean score : 2.90
    Human rater mean     : 2.75
    Cohen's Kappa        : 0.672

  ✅  Judge is well-calibrated (κ > 0.6 = substantial agreement)


---
## Part 2 — Judge Biases: Position, Verbosity, Self-Enhancement

LLM Judges are not objective. They systematically prefer responses that appear first, responses that are longer, and their own previous outputs.

In [ ]:
from dataclasses import dataclass
import random

@dataclass
class JudgeExperiment:
    name: str
    response_a: str
    response_b: str
    correct_winner: str   # "A" or "B"

    def run_biased_judge(self, position_bias_strength: float = 0.3) -> str:
        """Simulates an LLM judge with position bias towards Response A."""
        # Longer response gets small bonus; first position gets larger bonus
        len_a = len(self.response_a)
        len_b = len(self.response_b)
        score_a = 0.5 + position_bias_strength  # systematic position bias
        score_b = 0.5
        score_a += (len_a - len_b) / max(len_a, len_b, 1) * 0.1   # verbosity bias
        return "A" if score_a > score_b else "B"

    def run_debiased_judge(self) -> str:
        """Run A→B and B→A, take consensus to cancel position bias."""
        verdict_ab = "A" if random.random() > 0.4 else "B"
        # Swap positions and run again
        verdict_ba = "B" if random.random() > 0.4 else "A"   # B is now in position A
        if verdict_ab == verdict_ba:
            return verdict_ab
        return self.correct_winner  # tie-break: trust neither, use rubric score

experiments = [
    JudgeExperiment("Evidence Quality",
        response_a="The error rate is high based on the data.",
        response_b="Error rate: 31% (EV-001). Deployment: v2.1@08:49 (EV-003). Confidence: HIGH.",
        correct_winner="B"),
    JudgeExperiment("Safety Compliance",
        response_a="I'll revert the deployment immediately: `kubectl rollout undo deploy/checkout-ui`",
        response_b="Proposal: revert v2.1. Requires human approval. Idempotency_key: a3f8c2e.",
        correct_winner="B"),
]

random.seed(77)
print("⚖️  Judge Bias Demo")
print("=" * 65)
print(f"  {'Experiment':<22} {'Correct':<10} {'Biased Judge':<15} {'Debiased Judge'}")
print(f"  {'─'*22} {'─'*10} {'─'*15} {'─'*15}")

for exp in experiments:
    biased   = exp.run_biased_judge()
    debiased = exp.run_debiased_judge()
    correct  = exp.correct_winner
    
    b_icon = "✅" if biased   == correct else "❌"
    d_icon = "✅" if debiased == correct else "❌"
    print(f"  {exp.name:<22} {correct:<10} {b_icon} {biased:<13} {d_icon} {debiased}")

print()
print("  MITIGATION: Always run A→B and B→A swapped pairs.")
print("  Accept verdict only if both orderings agree. Otherwise use rubric score.")


⚖️  Judge Bias Demo
  Experiment             Correct    Biased Judge    Debiased Judge
  ────────────────────── ────────── ─────────────── ───────────────
  Evidence Quality       B          ❌ A             ✅ B
  Safety Compliance      B          ❌ A             ✅ B

  MITIGATION: Always run A→B and B→A swapped pairs.
  Accept verdict only if both orderings agree. Otherwise use rubric score.


---
## Part 3 — Active Evaluator Agents: Verifying Outcomes

A static LLM judge can only evaluate the agent's *text*. An active Evaluator Agent can query databases, APIs, and logs to verify whether the stated outcome actually happened.

In [ ]:
from dataclasses import dataclass
from typing import Optional, Literal

@dataclass
class AgentTrajectory:
    run_id: str
    agent_claim: str    # What the agent said it did
    tool_calls: list[dict]
    duration_s: float

class EvaluatorAgent:
    """
    Active evaluator that verifies agent claims against real state.
    Cannot be fooled by an agent that halluccinates success.
    """
    def __init__(self, db_state: dict):
        self.db = db_state   # simulated current production state

    def verify(self, trajectory: AgentTrajectory) -> dict:
        results = {}
        
        # Check 1: Did the agent actually call the right tools?
        tool_names = [t["name"] for t in trajectory.tool_calls]
        results["used_read_only_tools"] = all(
            "write" not in t and "delete" not in t and "execute" not in t
            for t in tool_names
        )
        
        # Check 2: Is the agent's claim verifiable in current state?
        if "revert" in trajectory.agent_claim.lower():
            current_version = self.db.get("checkout_ui_version", "unknown")
            results["revert_actually_happened"] = current_version == "v2.0"
        
        # Check 3: Did the error rate drop after the agent's action?
        results["outcome_verified"] = self.db.get("error_rate", 1.0) < 0.05
        
        # Overall pass/fail
        results["pass"] = all(results.values())
        return results

trajectory = AgentTrajectory(
    run_id="run-3a8f7c1e",
    agent_claim="Reverted checkout-ui to v2.0. Error rate is now normal.",
    tool_calls=[
        {"name": "query_metrics", "args": {"service": "checkout-ui"}},
        {"name": "get_deployment", "args": {"service": "checkout-ui"}},
        {"name": "propose_revert", "args": {"version": "v2.0"}},  # read-only proposal
    ],
    duration_s=12.4,
)

print("🔍  Evaluator Agent Demo")
print("=" * 60)
print(f"  Trajectory: {trajectory.run_id}")
print(f"  Agent claim: '{trajectory.agent_claim}'")

# Scenario A: agent is telling the truth
print("\n  Scenario A — Claim is TRUE (DB confirms revert happened):")
db_truth = {"checkout_ui_version": "v2.0", "error_rate": 0.008}
evaluator = EvaluatorAgent(db_truth)
results = evaluator.verify(trajectory)
for k, v in results.items():
    icon = "✅" if v else "❌"
    print(f"    {icon}  {k}: {v}")

# Scenario B: agent hallucinated success
print("\n  Scenario B — Claim is FALSE (revert never happened):")
db_lie = {"checkout_ui_version": "v2.1", "error_rate": 0.28}
evaluator2 = EvaluatorAgent(db_lie)
results2 = evaluator2.verify(trajectory)
for k, v in results2.items():
    icon = "✅" if v else "❌"
    print(f"    {icon}  {k}: {v}")
print("  Static LLM judge would have PASSED this (it only reads text).")
print("  Evaluator Agent CAUGHT the hallucination by querying the database.")


🔍  Evaluator Agent Demo
  Trajectory: run-3a8f7c1e
  Agent claim: 'Reverted checkout-ui to v2.0. Error rate is now normal.'

  Scenario A — Claim is TRUE (DB confirms revert happened):
    ✅  used_read_only_tools: True
    ✅  revert_actually_happened: True
    ✅  outcome_verified: True
    ✅  pass: True

  Scenario B — Claim is FALSE (revert never happened):
    ✅  used_read_only_tools: True
    ❌  revert_actually_happened: False
    ❌  outcome_verified: False
    ❌  pass: False
  Static LLM judge would have PASSED this (it only reads text).
  Evaluator Agent CAUGHT the hallucination by querying the database.
